In [36]:
# Testing the geoapify replacement for google maps api since osm is washed
import pandas as pd
import requests
import time
import json

In [ ]:
df = pd.read_csv("data_2026-03-02_2026-03-08.csv")
# Filter out bad coordinates
df = df.dropna(subset = ['longitude', 'latitude'])
df = df[(df['longitude'] != 0) & (df['latitude'] != 0)]

# Clean and combine address data for later processing
df['street_name'] = df['street_name'].apply(lambda x : x.title())
df['address'] = df.apply(lambda x : str(x['house_number']) + ' ' + x['street_name'] ,axis = 1)

df = df.iloc[1:5]

df = df[
    ['inspection_type',
     'job_id',
     'job_progress',
     'house_number',
     'street_name',
     'address',
     'zip_code',
     'latitude',
     'longitude',
     'result',
     'approved_date',
     'nta'
     ]
]
df.head()

,inspection_type,job_id,job_progress,house_number,street_name,address,zip_code,latitude,longitude,result,approved_date,nta
2,Initial,PC8673353,1,88-47,187 Street,88-47 187 Street,11423.0,40.713865,-73.774740,Rat Activity,2026-03-03T14:36:31.000,Hollis
3,Initial,PC8671158,1,675,Riverside Drive,675 Riverside Drive,10031.0,40.826757,-73.952242,Failed for Rat Act,2026-03-04T09:46:11.000,Hamilton Heights-Sugar Hill
4,Initial,PC8672570,1,1603,President Street,1603 President Street,11213.0,40.667220,-73.935026,Failed for Other R,2026-03-03T08:42:22.000,Crown Heights (South)
5,Initial,PC8662766,1,768,Putnam Avenue,768 Putnam Avenue,11221.0,40.686198,-73.931088,Rat Activity,2026-03-03T10:14:31.000,Bedford-Stuyvesant (East)
6,Treatments,PC8589110,4,73,Steuben Street,73 Steuben Street,11205.0,40.695219,-73.963422,Bait applied,2026-03-02T13:38:21.000,Clinton Hill


In [ ]:
# import requests

# API methods

categories = "catering.restaurant,commercial.food_and_drink,catering.fast_food,catering.food_court,catering.bar"
longitude = str(-74.00961)
latitude = str(40.71525)
radius = "50"
max_locations_returned = "5"


# url = f"https://api.geoapify.com/v2/places?categories=catering.restaurant,commercial.food_and_drink,catering.fast_food,catering.food_court,catering.bar&filter=circle:-74.00961946500342,40.71525406412272,50&bias=proximity:-74.00961946500342,40.71525406412272&limit=5&apiKey={geoapify_key}"

url = f"https://api.geoapify.com/v2/places?categories={categories}&filter=circle:{longitude},{latitude},{radius}&bias=proximity:{longitude},{latitude}&limit={max_locations_returned}&apiKey={geoapify_key}"
payload = {}
headers = {}

response = requests.request("GET", url, headers=headers, data=payload)

print(response.text)


{"type":"FeatureCollection","features":[{"type":"Feature","properties":{"name":"JR Sushi 2","country":"United States","country_code":"us","state":"New York","county":"New York County","city":"New York","postcode":"10007","district":"Tribeca","suburb":"Manhattan","quarter":"Lower Manhattan","street":"West Broadway","housenumber":"86A","iso3166_2":"US-NY","lon":-74.009614,"lat":40.7152802,"state_code":"NY","formatted":"JR Sushi 2, 86A West Broadway, New York, NY 10007, United States of America","address_line1":"JR Sushi 2","address_line2":"86A West Broadway, New York, NY 10007, United States of America","categories":["catering","catering.restaurant","catering.restaurant.sushi"],"details":["details","details.catering","details.contact"],"datasource":{"sourcename":"openstreetmap","attribution":"© OpenStreetMap contributors","license":"Open Database License","url":"https://www.openstreetmap.org/copyright","raw":{"name":"JR Sushi 2","phone":"+1-212-233-8338","osm_id":2641379134,"amenity":"re

In [55]:
res = json.loads(response.text)

df = pd.DataFrame([
    {
        **feature["properties"],
        "geo_lon": feature["geometry"]["coordinates"][0],
        "geo_lat": feature["geometry"]["coordinates"][1],
    }
    for feature in res["features"]
])
print(df.columns)
selected_columns = ['name', 'county', 'city', 'postcode', 'district', 'suburb', 'quarter', 'street', 'housenumber','lon','lat', 'formatted', 'catering']
df = df[selected_columns]
df

Index(['name', 'country', 'country_code', 'state', 'county', 'city',
       'postcode', 'district', 'suburb', 'quarter', 'street', 'housenumber',
       'iso3166_2', 'lon', 'lat', 'state_code', 'formatted', 'address_line1',
       'address_line2', 'categories', 'details', 'datasource', 'website',
       'contact', 'catering', 'distance', 'place_id', 'geo_lon', 'geo_lat',
       'facilities', 'building'],
      dtype='object')


,name,county,city,postcode,district,suburb,quarter,street,housenumber,lon,lat,formatted,catering
0,JR Sushi 2,New York County,New York,10007,Tribeca,Manhattan,Lower Manhattan,West Broadway,86A,-74.009614,40.715280,"JR Sushi 2, 86A West Broadway, New York, NY 10...",{'cuisine': 'sushi'}
1,Galerie Bar,New York County,New York,10012,Tribeca,Manhattan,Lower Manhattan,West Broadway,NaN,-74.009289,40.715139,"Galerie Bar, West Broadway, New York, NY 10012...",NaN
2,The Smyth,New York County,New York,10012,Tribeca,Manhattan,Lower Manhattan,West Broadway,NaN,-74.009197,40.715238,"The Smyth, West Broadway, New York, NY 10012, ...",NaN
3,Mulberry & Vine,New York County,New York,10007,Tribeca,Manhattan,Lower Manhattan,Warren Street,73,-74.010209,40.714983,"Mulberry & Vine, 73 Warren Street, New York, N...",{'cuisine': 'american'}


In [ ]:

categories = "catering.restaurant,commercial.food_and_drink,catering.fast_food,catering.food_court,catering.bar"
radius = "50"
max_locations_returned = "5"

all_restaurants_df = pd.DataFrame()

for _ , row in df.iterrows():

    time.sleep(2)
    longitude = str(row['longitude'])
    latitude = str(row['latitude'])

    url = f"https://api.geoapify.com/v2/places?categories={categories}&filter=circle:{longitude},{latitude},{radius}&bias=proximity:{longitude},{latitude}&limit={max_locations_returned}&apiKey={geoapify_key}"

    payload = {}
    headers = {}

    response = requests.request("GET", url, headers=headers, data=payload)

    res = json.loads(response.text)
    print(res)
    try:
        local_restaurant_df = pd.DataFrame([
            {
                **feature["properties"],
                "geo_lon": feature["geometry"]["coordinates"][0],
                "geo_lat": feature["geometry"]["coordinates"][1],
            }
            for feature in res["features"]
        ])
        
        # selected_columns = ['name', 'county', 'city', 'postcode', 'district', 'suburb', 'quarter', 'street', 'housenumber','lon','lat', 'formatted', 'catering']
        # local_restaurant_df = local_restaurant_df[selected_columns]
        
        all_restaurants_df = pd.concat([all_restaurants_df, local_restaurant_df])
    except Exception as e:
        print(e)
        # Move on if no data is found
        continue
    

{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'name': 'Hello, Yam!', 'country': 'United States', 'country_code': 'us', 'state': 'New York', 'county': 'New York County', 'city': 'New York', 'postcode': '10009', 'district': 'East Village', 'suburb': 'Manhattan', 'street': 'East 9th Street', 'housenumber': '4432', 'iso3166_2': 'US-NY', 'lon': -73.9829136, 'lat': 40.727499, 'state_code': 'NY', 'formatted': 'Hello, Yam!, 4432 East 9th Street, New York, NY 10009, United States of America', 'address_line1': 'Hello, Yam!', 'address_line2': '4432 East 9th Street, New York, NY 10009, United States of America', 'categories': ['catering', 'catering.fast_food', 'vegan'], 'details': ['details', 'details.catering', 'details.contact'], 'datasou

In [84]:
print(all_restaurants_df.columns)
all_restaurants_df.head()

Index(['name', 'country', 'country_code', 'state', 'county', 'city',
       'postcode', 'district', 'suburb', 'street', 'housenumber', 'iso3166_2',
       'lon', 'lat', 'state_code', 'formatted', 'address_line1',
       'address_line2', 'categories', 'details', 'datasource', 'website',
       'contact', 'catering', 'distance', 'place_id', 'geo_lon', 'geo_lat',
       'opening_hours', 'description', 'commercial', 'facilities', 'brand',
       'brand_details', 'quarter', 'building', 'name_other'],
      dtype='object')


,name,country,country_code,state,county,city,postcode,district,suburb,street,...,geo_lat,opening_hours,description,commercial,facilities,brand,brand_details,quarter,building,name_other
0,"Hello, Yam!",United States,us,New York,New York County,New York,10009,East Village,Manhattan,East 9th Street,...,40.727499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Confectionery!,United States,us,New York,New York County,New York,10003,East Village,Manhattan,East 9th Street,...,40.727320,"Mo-Su 12:00-15:30, Mo-Sa 16:15-20:00",Vegan sweet shop.,{'type': 'bakery'},NaN,NaN,NaN,NaN,NaN,NaN
2,Doc Hollidays,United States,us,New York,New York County,New York,10009,Alphabet City,Manhattan,Avenue A,...,40.727228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Poke N' Roll,United States,us,New York,New York County,New York,10009,East Village,Manhattan,East 9th Street,...,40.727541,Mo-Su 11:00-22:00,NaN,NaN,{'takeaway': True},NaN,NaN,NaN,NaN,NaN
4,Tacos Morelos,United States,us,New York,New York County,New York,10009,East Village,Manhattan,East 9th Street,...,40.727363,Mo-Su 11:00-22:30,NaN,NaN,"{'takeaway': True, 'delivery': True}",NaN,NaN,NaN,NaN,NaN


In [86]:
selected_columns = ['name', 'county', 'city', 'postcode', 'district', 'suburb', 'housenumber', 'street', 'lon','lat', 'formatted', 'catering', 'commercial']
geo_rest_df = all_restaurants_df[selected_columns]

In [88]:
pd.merge(left = df, right = geo_rest_df, how = "inner", left_on = ['house_number', 'street_name'], right_on = ['housenumber', 'street'])

,inspection_type,job_id,job_progress,house_number,street_name,address,zip_code,latitude,longitude,result,...,postcode,district,suburb,housenumber,street,lon,lat,formatted,catering,commercial
0,Treatments,PC8488952,7,584,Myrtle Avenue,584 Myrtle Avenue,11205.0,40.693975,-73.961259,Bait applied,...,11205,NaN,Brooklyn,584,Myrtle Avenue,-73.961238,40.693875,"Tipsy, 584 Myrtle Avenue, New York, NY 11205, ...",NaN,{'type': 'alcohol'}
1,Initial,PC8666642,1,188,Wilson Avenue,188 Wilson Avenue,11237.0,40.699379,-73.923425,Rat Activity,...,11207,NaN,Brooklyn,188,Wilson Avenue,-73.923511,40.699303,"Happy Garden, 188 Wilson Avenue, New York, NY ...",{'cuisine': 'chinese'},NaN
2,Initial,PC8666642,1,188,Wilson Avenue,188 Wilson Avenue,11237.0,40.699379,-73.923425,Rat Activity,...,11207,NaN,Brooklyn,188,Wilson Avenue,-73.923511,40.699303,"Happy Garden, 188 Wilson Avenue, New York, NY ...",{'cuisine': 'chinese'},NaN
